# 83 — Simulate10Next: Conqueror & Supplier Tests

Test notebook for `StrategyPipeline` from `82-Simulate10Next_Conqueror_Supplier.py`.
Each test exposes `df_s`, `pa`, `safe`, and `action` for interactive inspection.

In [1]:
%run 82-Simulate10Next_Conqueror_Supplier.py

In [2]:
import copy, math, random
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
# import kaggle_environments as ke

_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


def simulate_with_action(obs, action0, n_steps, current_step=0):
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def make_animation(snapshots, title='', interval=150):
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y,   str(ships),         ha='center', va='center', color='white', fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y+2, str(pid),            ha='center', va='center', color='red',   fontsize=7, fontweight='bold', zorder=4)
            ax.text(x, y-2, "+"+str(production), ha='center', va='center', color='white', fontsize=5, fontweight='bold', zorder=4)
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)
        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

## Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy

Planet 1 (x=30) should attack planet 2 (x=70). Planet 0 (x=20) stays as supplier.

In [3]:
obs01 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50, 3],  # Supplier
        [1, 0, 25.0, 10.0, 1 + math.log(3), 50, 3],  # Conqueror
        [2, 1, 75.0, 10.0, 1.0,              1,  1],  # Enemy
    ],
    angular_velocity=0.05,
)
df_s01, pd01 = StrategyPipeline._01_get_obs_dataframe(obs01, step=0, num_agents=2)
df_s01

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,20.0,20.0,2.098612,50,3,0,moving
1,0,1,30.0,20.0,2.098612,50,3,0,moving
2,0,2,70.0,20.0,1.000000,1,1,1,moving
3,1,0,20.0,20.0,2.098612,53,3,0,moving
4,1,1,30.0,20.0,2.098612,53,3,0,moving
5,1,2,70.0,20.0,1.000000,2,1,1,moving
6,2,0,20.0,20.0,2.098612,56,3,0,moving
7,2,1,30.0,20.0,2.098612,56,3,0,moving
8,2,2,70.0,20.0,1.000000,3,1,1,moving
9,3,0,20.0,20.0,2.098612,59,3,0,moving


In [4]:
pa01 = StrategyPipeline._02_get_all_opportunities(df_s01, pd01, player_id=0)
pa01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,-3.552714e-16
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,-3.552714e-16
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,7.901388,8.019942,0.000000,0.077668,6.205517,0.077668,-3.552714e-16
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,7.901388,8.060946,0.000000,0.089429,6.193757,0.089429,-3.552714e-16
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,7.901388,8.101053,0.000000,0.099302,6.183884,0.099302,-3.552714e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,1,0,30.0,20.0,2.098612,50,3,moving,0,11,...,-3.141593e+00,10.0,10.0,10.198612,11.198612,0.207246,0.162965,2.934347,3.348838,-3.141593e+00
426,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,10.198612,11.198612,0.207246,0.162965,6.075940,0.207246,-3.552714e-16
427,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,11.470043,12.098612,0.139959,0.000000,6.143226,0.139959,-3.552714e-16
428,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.0,10.0,11.198612,12.098612,0.162965,0.000000,6.120220,0.162965,-3.552714e-16


In [5]:
safe01 = StrategyPipeline._03_filter_collision(pa01)
safe01

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,-3.552714e-16,-3.552714e-16
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,-3.552714e-16,-3.552714e-16
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,7.901388,8.019942,0.000000,0.077668,6.205517,0.077668,-3.552714e-16,-3.552714e-16
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,7.901388,8.060946,0.000000,0.089429,6.193757,0.089429,-3.552714e-16,-3.552714e-16
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,7.901388,8.101053,0.000000,0.099302,6.183884,0.099302,-3.552714e-16,-3.552714e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,1,0,30.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,10.198612,11.198612,0.207246,0.162965,2.934347,3.348838,-3.141593e+00,-3.141593e+00
426,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,10.198612,11.198612,0.207246,0.162965,6.075940,0.207246,-3.552714e-16,-3.552714e-16
427,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,11.470043,12.098612,0.139959,0.000000,6.143226,0.139959,-3.552714e-16,-3.552714e-16
428,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.0,10.0,11.198612,12.098612,0.162965,0.000000,6.120220,0.162965,-3.552714e-16,-3.552714e-16


In [6]:
action01 = StrategyPipeline._04_score_and_decide(safe01, player_id=0)
print("Action:", action01)
snaps01 = simulate_with_action(copy.deepcopy(obs01), action01, 20)
make_animation(snaps01, title='Test 01 — 1 Supplier, 1 Conqueror, 1 Enemy', interval=200)

Action: []


## Test 02 — 1 Supplier, 2 Conquerors (one more in need)

Planet 1 attacks planet 2 (easy). Planet 3 attacks planet 4 (heavy — 100 ships). Planet 0 stays as supplier.

In [7]:
obs02 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50,  3],
        [1, 0, 25.0, 10.0, 1 + math.log(3), 50,  3],
        [2, 1, 75.0, 10.0, 1 + math.log(3), 1,   1],
        [3, 0, 10.0, 25.0, 1 + math.log(3), 50,  3],
        [4, 1, 10.0, 75.0, 1 + math.log(3), 100, 1],
    ],
    angular_velocity=0.05,
)
df_s02, pd02 = StrategyPipeline._01_get_obs_dataframe(obs02, step=0, num_agents=2)
df_s02

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,20.0,20.0,2.098612,50,3,0,moving
1,0,1,30.0,20.0,2.098612,50,3,0,moving
2,0,2,70.0,20.0,2.098612,1,1,1,moving
3,0,3,20.0,30.0,2.098612,50,3,0,moving
4,0,4,20.0,70.0,2.098612,100,1,1,moving
5,1,0,20.0,20.0,2.098612,53,3,0,moving
6,1,1,30.0,20.0,2.098612,53,3,0,moving
7,1,2,70.0,20.0,2.098612,2,1,1,moving
8,1,3,20.0,30.0,2.098612,53,3,0,moving
9,1,4,20.0,70.0,2.098612,101,1,1,moving


In [8]:
pa02 = StrategyPipeline._02_get_all_opportunities(df_s02, pd02, player_id=0)
pa02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.000000,10.000000,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,-3.552714e-16
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.000000,10.000000,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,-3.552714e-16
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.000000,10.000000,7.901388,8.019942,0.000000,0.077668,6.205517,0.077668,-3.552714e-16
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.000000,10.000000,7.901388,8.060946,0.000000,0.089429,6.193757,0.089429,-3.552714e-16
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,-3.552714e-16,10.000000,10.000000,7.901388,8.101053,0.000000,0.099302,6.183884,0.099302,-3.552714e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1271,1,0,30.0,20.0,2.098612,50,3,moving,0,11,...,2.356194e+00,14.142136,14.142136,15.244281,16.240748,0.121707,0.000000,2.234488,2.477901,2.356194e+00
1272,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,-1.570796e+00,10.000000,10.000000,11.198612,12.098612,0.162965,0.000000,4.549424,4.875354,-1.570796e+00
1273,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,-7.853982e-01,14.142136,14.142136,12.043523,12.198612,0.000000,0.060291,5.437497,5.558078,-7.853982e-01
1274,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,-7.853982e-01,14.142136,14.142136,14.052742,15.369868,0.148868,0.115508,5.348919,5.646655,-7.853982e-01


In [9]:
safe02 = StrategyPipeline._03_filter_collision(pa02)
safe02

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,7.935062,0.000000,0.042038,6.241148,0.042038,-3.552714e-16,-3.552714e-16
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,7.977996,0.000000,0.062914,6.220272,0.062914,-3.552714e-16,-3.552714e-16
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.019942,0.000000,0.077668,6.205517,0.077668,-3.552714e-16,-3.552714e-16
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.060946,0.000000,0.089429,6.193757,0.089429,-3.552714e-16,-3.552714e-16
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.101053,0.000000,0.099302,6.183884,0.099302,-3.552714e-16,-3.552714e-16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1271,1,0,30.0,20.0,2.098612,50,3,moving,0,11,...,14.142136,14.142136,15.244281,16.240748,0.121707,0.000000,2.234488,2.477901,2.356194e+00,2.356194e+00
1272,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,11.198612,12.098612,0.162965,0.000000,4.549424,4.875354,-1.570796e+00,-1.570796e+00
1273,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,14.142136,14.142136,12.043523,12.198612,0.000000,0.060291,5.437497,5.558078,-7.853982e-01,-7.853982e-01
1274,3,0,20.0,30.0,2.098612,50,3,moving,0,11,...,14.142136,14.142136,14.052742,15.369868,0.148868,0.115508,5.348919,5.646655,-7.853982e-01,-7.853982e-01


In [10]:
action02 = StrategyPipeline._04_score_and_decide(safe02, player_id=0)
print("Action:", action02)
snaps02 = simulate_with_action(copy.deepcopy(obs02), action02, 20)
make_animation(snaps02, title='Test 02 — 1 Supplier, 2 Conquerors (one more in need)', interval=200)

Action: []


## Test 03 — 4 Suppliers, 2 Conquerors (one more in need)

Same enemies as Test 02. Planets 5, 6, 7 added as extra suppliers. Pipeline should still route correctly.

In [11]:
obs03 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50,  3],
        [1, 0, 25.0, 10.0, 1 + math.log(3), 50,  3],
        [2, 1, 75.0, 10.0, 1 + math.log(3), 1,   1],
        [3, 0, 10.0, 25.0, 1 + math.log(3), 50,  3],
        [4, 1, 10.0, 75.0, 1 + math.log(3), 100, 1],
        [5, 0,  5.0,  5.0, 1 + math.log(3), 50,  3],
        [6, 0,  5.0, 15.0, 1 + math.log(3), 50,  3],
        [7, 0, 15.0,  5.0, 1 + math.log(3), 50,  3],
    ],
    angular_velocity=0.05,
)
df_s03, pd03 = StrategyPipeline._01_get_obs_dataframe(obs03, step=0, num_agents=2)
df_s03

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,20.0,20.0,2.098612,50,3,0,moving
1,0,1,30.0,20.0,2.098612,50,3,0,moving
2,0,2,70.0,20.0,2.098612,1,1,1,moving
3,0,3,20.0,30.0,2.098612,50,3,0,moving
4,0,4,20.0,70.0,2.098612,100,1,1,moving
...,...,...,...,...,...,...,...,...,...
83,10,3,20.0,30.0,2.098612,80,3,0,moving
84,10,4,20.0,70.0,2.098612,110,1,1,moving
85,10,5,10.0,10.0,2.098612,80,3,0,fix
86,10,6,10.0,25.0,2.098612,80,3,0,moving


In [12]:
pa03 = StrategyPipeline._02_get_all_opportunities(df_s03, pd03, player_id=0)
pa03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,1.570796,10.000000,10.000000,7.901388,8.060946,0.000000,8.942875e-02,1.481368,1.660225,1.570796
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,1.570796,10.000000,10.000000,7.901388,8.101053,0.000000,9.930158e-02,1.471495,1.670098,1.570796
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,1.570796,10.000000,10.000000,7.901388,8.140301,0.000000,1.078365e-01,1.462960,1.678633,1.570796
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,1.570796,10.000000,10.000000,7.901388,8.178730,0.000000,1.153567e-01,1.455440,1.686153,1.570796
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,1.570796,10.000000,10.000000,7.901388,8.216375,0.000000,1.220722e-01,1.448724,1.692869,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,2.356194,21.213203,21.213203,21.380429,23.311816,0.098268,0.000000e+00,2.257926,2.454463,2.356194
5836,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,2.356194,21.213203,21.213203,21.824870,23.311816,0.093333,0.000000e+00,2.262862,2.449527,2.356194
5837,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,2.356194,21.213203,21.213203,22.244285,23.311816,0.084170,0.000000e+00,2.272025,2.440364,2.356194
5838,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,2.356194,21.213203,21.213203,22.641491,23.311816,0.070173,0.000000e+00,2.286021,2.426368,2.356194


In [13]:
safe03 = StrategyPipeline._03_filter_collision(pa03)
safe03

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.060946,0.000000,8.942875e-02,1.481368,1.660225,1.570796,1.570796
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.101053,0.000000,9.930158e-02,1.471495,1.670098,1.570796,1.570796
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.140301,0.000000,1.078365e-01,1.462960,1.678633,1.570796,1.570796
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.178730,0.000000,1.153567e-01,1.455440,1.686153,1.570796,1.570796
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,10.000000,10.000000,7.901388,8.216375,0.000000,1.220722e-01,1.448724,1.692869,1.570796,1.570796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,21.213203,21.213203,21.380429,23.311816,0.098268,0.000000e+00,2.257926,2.454463,2.356194,2.356194
5836,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,21.213203,21.213203,21.824870,23.311816,0.093333,0.000000e+00,2.262862,2.449527,2.356194,2.356194
5837,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,21.213203,21.213203,22.244285,23.311816,0.084170,0.000000e+00,2.272025,2.440364,2.356194,2.356194
5838,7,0,25.0,10.0,2.098612,50,3,moving,0,11,...,21.213203,21.213203,22.641491,23.311816,0.070173,0.000000e+00,2.286021,2.426368,2.356194,2.356194


In [14]:
action03 = StrategyPipeline._04_score_and_decide(safe03, player_id=0)
print("Action:", action03)
snaps03 = simulate_with_action(copy.deepcopy(obs03), action03, 20)
make_animation(snaps03, title='Test 03 — 4 Suppliers, 2 Conquerors (one more in need)', interval=200)

Action: []


## Test 04 — 4 Suppliers, 1 Conqueror Orbiting

Planets 4 and 5 orbit (dist_from_center < 50 − radius). `angular_velocity=0.05`. The pipeline must intercept the moving enemy planet.

In [15]:
obs04 = Obs(
    planets=[
        [0, 0, 10.0, 10.0, 1 + math.log(3), 50, 3],
        [1, 0,  5.0,  5.0, 1 + math.log(3), 50, 3],
        [2, 0,  5.0, 15.0, 1 + math.log(3), 50, 3],
        [3, 0, 15.0,  5.0, 1 + math.log(3), 50, 3],
        [4, 0, 15.0, 15.0, 1.0,              50, 1],  # static (dist≈49.5, just outside orbit)
        [5, 1, 25.0, 25.0, 1.0,              50, 1],  # orbiting enemy (dist≈35 < 49)
    ],
    angular_velocity=0.05,
)
df_s04, pd04 = StrategyPipeline._01_get_obs_dataframe(obs04, step=0, num_agents=2)
df_s04

,step,id,x,y,radius,ships,production,owner,nature
0,0,0,20.000000,20.000000,2.098612,50,3,0,moving
1,0,1,10.000000,10.000000,2.098612,50,3,0,fix
2,0,2,10.000000,25.000000,2.098612,50,3,0,moving
3,0,3,25.000000,10.000000,2.098612,50,3,0,moving
4,0,4,25.000000,25.000000,1.000000,50,1,0,moving
...,...,...,...,...,...,...,...,...,...
61,10,1,10.000000,10.000000,2.098612,80,3,0,fix
62,10,2,24.856254,10.090201,2.098612,80,3,0,moving
63,10,3,44.887444,3.107978,2.098612,80,3,0,moving
64,10,4,38.362961,16.614684,1.000000,60,1,0,moving


In [16]:
pa04 = StrategyPipeline._02_get_all_opportunities(df_s04, pd04, player_id=0)
pa04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,angle_t2,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,2.806261,9.690853,9.213725,7.616528,8.393914,0.037058,2.201166e-01,2.586144,3.026377,2.787732
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,2.806261,9.679785,9.213725,7.604344,8.359736,0.036251,2.188641e-01,2.587397,3.025125,2.788135
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,2.806261,9.668428,9.213725,7.591869,8.324920,0.035420,2.174977e-01,2.588763,3.023758,2.788551
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,2.806261,9.656770,9.213725,7.579092,8.289440,0.034564,2.160084e-01,2.590252,3.022269,2.788979
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,2.806261,9.644794,9.213725,7.565997,8.253267,0.033682,2.143862e-01,2.591875,3.020647,2.789420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3273,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,-1.580437,13.765464,14.910492,12.954130,14.271256,0.145064,1.371360e-01,4.413658,4.839884,-1.652450
3274,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,-0.108446,14.417975,15.783086,15.160791,16.723102,0.045287,2.099783e-02,6.153742,6.244479,-0.096219
3275,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,-2.356194,21.213203,21.213203,23.266971,23.311816,0.019424,1.490116e-08,3.907567,3.946415,-2.356194
3276,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,-2.356194,21.213203,21.213203,22.279730,23.311816,0.083161,1.490116e-08,3.843830,4.010152,-2.356194


In [17]:
safe04 = StrategyPipeline._03_filter_collision(pa04)
safe04

,id_src,step_src,x_src,y_src,radius_src,ships_min,production_src,nature_src,owner_src,row_count,...,d_s_t1,d_s_t2,d_f_t1,d_f_t2,angle_radius_t1,angle_radius_t2,angle_min,angle_max,angle,final_angle
0,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,9.690853,9.213725,7.616528,8.393914,0.037058,0.220117,2.586144,3.026377,2.787732,2.787732
1,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,9.679785,9.213725,7.604344,8.359736,0.036251,0.218864,2.587397,3.025125,2.788135,2.788135
2,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,9.668428,9.213725,7.591869,8.324920,0.035420,0.217498,2.588763,3.023758,2.788551,2.788551
3,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,9.656770,9.213725,7.579092,8.289440,0.034564,0.216008,2.590252,3.022269,2.788979,2.788979
4,0,0,20.0,20.0,2.098612,50,3,moving,0,11,...,9.644794,9.213725,7.565997,8.253267,0.033682,0.214386,2.591875,3.020647,2.789420,2.789420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999,2,0,10.0,25.0,2.098612,50,3,moving,0,11,...,30.312877,30.737873,29.321573,30.204573,0.004414,0.027763,6.199818,6.255345,-0.053397,-0.053397
3000,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,13.765464,14.910492,14.145668,15.595187,0.148039,0.130184,4.410684,4.832932,-1.652450,-1.652450
3001,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,13.765464,14.910492,12.954130,14.271256,0.145064,0.137136,4.413658,4.839884,-1.652450,-1.652450
3002,4,0,25.0,25.0,1.000000,50,1,moving,0,11,...,14.417975,15.783086,15.160791,16.723102,0.045287,0.020998,6.153742,6.244479,-0.096219,-0.096219


In [18]:
action04 = StrategyPipeline._04_score_and_decide(safe04, player_id=0)
print("Action:", action04)
snaps04 = simulate_with_action(copy.deepcopy(obs04), action04, 20)
make_animation(snaps04, title='Test 04 — 4 Suppliers, 1 Conqueror Orbiting', interval=200)

Action: []


## Test 05 — Our Agent vs Random Agent

Full game via `kaggle_environments`. Player 0 uses `agent()` from `82-...py`, player 1 uses a random policy.

In [19]:
def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]


# Reset agent globals so this cell is re-runnable
step = 0
num_agents = None
player_id = None

SEED = 42
N_STEPS = 100
random.seed(SEED)

env = ke.make("orbit_wars", debug=False)
env.reset(2)

snaps05 = []
for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation
    snaps05.append({
        'step':    env_step,
        'planets': [list(p) for p in obs0.planets],
        'fleets':  [list(f) for f in obs0.fleets],
    })
    action0 = agent(obs0)
    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

obs0 = env.state[0].observation
snaps05.append({
    'step':    len(snaps05),
    'planets': [list(p) for p in obs0.planets],
    'fleets':  [list(f) for f in obs0.fleets],
})
p0 = sum(p[5] for p in obs0.planets if p[1] == 0)
p1 = sum(p[5] for p in obs0.planets if p[1] == 1)
winner = "Our agent wins" if p0 > p1 else "Random wins" if p1 > p0 else "Tie"
print(f"After {len(snaps05) - 1} steps: {winner}  (player0={p0}, player1={p1})")

make_animation(snaps05, title='Test 05 — Our Agent vs Random', interval=100)

NameError: name 'ke' is not defined